# CISC 451 — A1: RQ1, RQ2, RQ3 (executable notebook)

This notebook **runs the actual pipeline live** (model inference, reasoning-trace analysis, and the
PlanPlay-SQL LoRA fine-tuning chain) and recomputes every table/figure in the report
([`report/main.tex`](report/main.tex), built from [`report/RQ1.tex`](report/RQ1.tex) / [`report/RQ2.tex`](report/RQ2.tex) / [`report/RQ3.tex`](report/RQ3.tex))
from that run, rather than reading the committed result files. It does not call any external API — everything
below runs locally against the HuggingFace checkpoints already cached on this machine.

**Read this before running:**

- **Setup.** See [`README.md`](README.md) for one-time repo setup (submodules, mock/schema SQLite DBs,
  Python deps, HF cache). Run the setup steps there first — §0 below only re-checks that setup, it does not
  perform it from scratch.
- **Runtime.** This is a long notebook. RQ1 generation (§1) takes roughly 1 hour total for all three models
  on a 24 GB GPU; the full RQ3 fine-tuning chain (§3) took several hours on the 8 GB GPU this repo was
  developed on (see the historical logs in `train/runs/*.log` / `*.out`). Every heavy cell is **idempotent**:
  it checks whether its output already exists and skips the work if so, so re-running the notebook top to
  bottom after a partial run only does the remaining work.
- **Fresh run, new paths.** Every artifact this notebook produces is written under a `*_nb` / `rq1_notebook`
  suffix (e.g. `eval/results/Qwen__Qwen3-0.6B/rq1_notebook/`, `train/runs/e1_plan_nb/`). Nothing under this
  notebook's control overwrites the artifacts the report's own numbers were generated from.
- **Numbers will differ slightly from the report.** This machine's GPU, and the exact point in the RNG
  stream when each cell runs, differ from the original report run; `RQ1.tex`/`RQ2.tex`/`RQ3.tex`'s own
  Limitations sections already flag that single-run, single-seed results like these are not expected to
  reproduce to the third decimal place. The qualitative conclusions should match.
- **No Qwen3-4B extension.** `RQ3.tex` scopes RQ3 to Qwen3-0.6B only (no Qwen3-4B rerun), so this notebook
  does not run one either.

Section numbers below (§1.1, §2.3, Table 7, ...) are this notebook's own internal numbering, carried
over from an earlier markdown draft of the report. `RQ1.tex` / `RQ2.tex` / `RQ3.tex` have since been
restructured into named subsections and no longer use this numbering; treat a §-reference below as a
pointer to the relevant discussion in those files, not an exact section number.

## Imports and environment check

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
assert (REPO_ROOT / "README.md").exists() and (REPO_ROOT / "eval").exists(), (
    "Run this notebook with its working directory set to the repo root "
    "(the directory containing README.md and eval/)."
)
sys.path.insert(0, str(REPO_ROOT))

import torch
print(f"torch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}, "
          f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB total VRAM")
else:
    print("No GPU detected -- generation and training below will be extremely slow or infeasible.")

In [ ]:
def sh(args, env_extra=None, cwd=REPO_ROOT):
    """Run a subprocess with argv `args` (a list, never a shell string), streaming output live."""
    env = {**os.environ, **(env_extra or {})}
    print("$", " ".join(str(a) for a in args))
    t0 = time.time()
    subprocess.run(args, check=True, env=env, cwd=cwd)
    print(f"  ({time.time() - t0:.0f}s)")


def py(code_str, env_extra=None, cwd=REPO_ROOT):
    sh([sys.executable, "-c", code_str], env_extra, cwd)


def ensure(marker: Path, args, env_extra=None, note=""):
    """Run `args` unless `marker` already exists -- makes every heavy cell safe to re-run."""
    if marker.exists():
        print(f"[skip] {marker} already exists. {note}")
        return
    sh(args, env_extra)

## 0. One-time data setup (idempotent)

Two SQLite database trees are needed:

- `spider/database/<db_id>/<db_id>.sqlite` -- **schema-only** (no rows). Used for RQ1 (§1) and RQ2 (§2);
  with no rows, execution accuracy there is schema-level only (`RQ1.tex`).
- `mock_spider/database/<db_id>/<db_id>.sqlite` -- schema **plus** synthetic rows seeded from the literals
  in each question's gold query. Used for RQ3 (§3), since the official Spider row data was not available
  (`RQ3.tex` Limitations).

Both builders are idempotent (`build_spider_dbs.py` skips existing files; `train.mockdb` rebuilds
deterministically from the same seed), so this cell is safe to re-run.

In [ ]:
from eval.evalpipeline.common import DB_DIR as SCHEMA_DB_DIR, RESULTS_DIR, load_model_configs

if not (REPO_ROOT / "spider" / "evaluation_examples" / "examples" / "tables.json").exists():
    raise FileNotFoundError(
        "spider/ is missing tables.json -- see README.md section 1 (Spider dataset files)."
    )

sh([sys.executable, "build_spider_dbs.py"])
sh([sys.executable, "-m", "train.mockdb"], {"SPIDER_DB_DIR": "mock_spider/database"})
print("schema-only DBs:", len(list((REPO_ROOT / 'spider' / 'database').iterdir())))
print("mock DBs:       ", len(list((REPO_ROOT / 'mock_spider' / 'database').iterdir())))

# Task 1 -- RQ1: Methodology

RQ1: How do (selected) LLMs perform on cross-domain Text-to-SQL?

Full methodology: `RQ1.tex`. The cell below reproduces the model table directly from
`eval/configs/models.yaml`, so this notebook and `models.yaml` can never silently drift apart.

In [ ]:
MODEL_META = {  # presentational only: parameter count / category, not used in any computation below
    "Qwen/Qwen3-0.6B": ("0.6B", "Decoder-only"),
    "Qwen/Qwen3-4B": ("4B", "Decoder-only"),
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B": ("1.5B", "Reasoning"),
}
RQ1_MODELS = list(MODEL_META)  # the three models compared in RQ1.tex's tab:models

cfgs = load_model_configs()
rows = []
for repo_id in RQ1_MODELS:
    cfg = cfgs[repo_id]
    params, category = MODEL_META[repo_id]
    rows.append({"Model": repo_id, "Category": category, "Parameters": params,
                 "Quantization": cfg.get("quantization", "nf4"), "Thinking mode": cfg.get("thinking", "N/A"),
                 "max_new_tokens": cfg["max_new_tokens"]})
pd.DataFrame(rows)

**Local checkpoints.** All weight loading in this repo (RQ1 generation, RQ3 fine-tuning/evaluation) goes through `eval.evalpipeline.common.resolve_local_path`, which checks `./models/<author>/<name>` (or `$MODELS_PATH` if set) first and only falls back to the HF cache -- downloading on first use if not already cached -- for whatever isn't found there. The cell below just reports that check up front so it's clear before any heavy cell runs which models are already local.

In [ ]:
from eval.evalpipeline.common import MODELS_DIR, _local_dir

print(f"Local checkpoint directory: {MODELS_DIR}\n")
for repo_id in RQ1_MODELS:
    local = _local_dir(repo_id, MODELS_DIR)
    status = f"found at {local}" if local else "not found locally -- will use/download the HF cache"
    print(f"  {repo_id}: {status}")


In [ ]:
from eval.evalpipeline.common import TEMPLATE, build_user_prompt
print(TEMPLATE)

## 1.3-1.6 -- Live generation + evaluation (all three models, full 1034-example dev split)

This calls the exact entry point RQ1.tex's numbers came from: `python -m eval.evalpipeline.run`
(see `eval/evalpipeline/run.py`), with `--etype all` so the saved `metrics.json` has component matching,
exact matching, and execution accuracy together, exactly as parsed below.

**Batch size.** The original run used batch size 128 on a 24 GB RTX 4090. Greedy decoding makes the batch
size a pure speed/memory knob -- it does not change which tokens are generated -- so it is safe to lower it
for a smaller GPU. `GEN_BATCH_SIZE` below is set per model (128, or 64 for the reasoning model's long traces), sized for an 80 GB GPU;
lower it if `torch.cuda.get_device_properties(0).total_memory` printed above is much smaller.

**Runtime.** With batch size 128 on an RTX 4090, §2.5 (Table 9) reports ~124s + 45s + 3060s of raw
generation time (~54 minutes total) plus evaluation overhead. A smaller batch size and/or smaller GPU will
take longer, particularly for the reasoning model (4096-token budget).

This cell is resumable: `--resume` continues an interrupted run from its last-written example, and the
whole cell is a no-op on a second run once every model's `generations.jsonl` is complete (the underlying
`generate()` loop skips ids already written).

In [ ]:
GEN_BATCH_SIZE = {  # per model; sized for an 80 GB H100 -- lower these on a smaller GPU
    "Qwen/Qwen3-0.6B": 128,
    "Qwen/Qwen3-4B": 128,
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B": 64,  # long reasoning traces need more KV-cache per sequence
}
RQ1_RUN_ID = "rq1_notebook"

for m in RQ1_MODELS:
    sh([
        sys.executable, "-m", "eval.evalpipeline.run",
        "--models", m,
        "--etype", "all",
        "--batch-size", str(GEN_BATCH_SIZE[m]),
        "--run-id", RQ1_RUN_ID,
        "--resume",
    ])

In [ ]:
RQ1_RUN_DIRS = {m: RESULTS_DIR / m.replace("/", "__") / RQ1_RUN_ID for m in RQ1_MODELS}
RQ1_METRICS = {m: json.loads((d / "metrics.json").read_text()) for m, d in RQ1_RUN_DIRS.items()}
for m, d in RQ1_RUN_DIRS.items():
    print(m, "->", d)

## 1.* -- A single per-question dataframe

Tables 1-4 (component matching / exact matching / execution accuracy, aggregated and by difficulty) are
read straight from each model's official `metrics.json` below, i.e. the same parsed `evaluation.py` output
`RQ1.tex` itself reports.

Tables 5-13 need *per-question* granularity (by database, by generation outcome, by token count, ...) that
`metrics.json` does not carry. For those we reuse `eval.evalpipeline.reasoning_metrics.per_question_eval`
(the same in-process Spider scorer RQ2's own analysis uses -- see `RQ2.tex`) to build one dataframe per
model with one row per dev question, then derive every remaining table with plain `pandas` groupbys instead
of re-deriving the scoring logic.

In [ ]:
from eval.evalpipeline import breakdown
from eval.evalpipeline.reasoning_metrics import load_rows, per_question_eval

def model_dataframe(model: str, run_dir: Path) -> pd.DataFrame:
    import yaml
    rows = load_rows(run_dir)
    limit = yaml.safe_load((run_dir / "config_used.yaml").read_text())["model"]["max_new_tokens"]
    scored = {s["id"]: s for s in per_question_eval(rows)}
    by_id = {r["id"]: r for r in rows}
    recs = []
    for qid, s in scored.items():
        r = by_id[qid]
        recs.append({
            "model": model, "id": qid, "db_id": r["db_id"], "new_tokens": r["new_tokens"],
            "outcome": breakdown.outcome(r, limit), "raw": r["raw"],
            **s,
        })
    df = pd.DataFrame(recs)
    df["difficulty"] = pd.Categorical(df["difficulty"], categories=["easy", "medium", "hard", "extra"], ordered=True)
    return df

rq1_df = pd.concat([model_dataframe(m, d) for m, d in RQ1_RUN_DIRS.items()], ignore_index=True)
rq1_df.head()

# Task 1 -- RQ1: Results

## 2.1 -- Table 1: overall performance summary

Component matching is the macro-average (mean accuracy / mean recall across the 10 Spider components, F1
from those two means) over the official per-component `metrics.json` scores -- the same definition
`RQ1.tex`'s tab:models uses.

In [ ]:
def overall_performance_summary() -> pd.DataFrame:
    rows = []
    for m in RQ1_MODELS:
        met = RQ1_METRICS[m]
        a, r, f = macro_component(met, "all")
        params, category = MODEL_META[m]
        rows.append({
            "Model": m, "Parameters": params, "Category": category,
            "Component matching (acc/rec/F1)": f"{a:.3f} / {r:.3f} / {f:.3f}",
            "Exact matching": met["exact matching accuracy"]["exact match"]["all"],
            "Execution accuracy": met["execution accuracy"]["execution"]["all"],
        })
    return pd.DataFrame(rows)

overall_performance_summary()

## 2.1-2.2 -- Tables 2-4: component matching / exact matching / execution accuracy by difficulty

In [ ]:
LEVELS = ["easy", "medium", "hard", "extra", "all"]

def table_component_by_difficulty() -> pd.DataFrame:
    rows = {"Problems (n)": {l: RQ1_METRICS[RQ1_MODELS[0]]["count"]["count"][l] for l in LEVELS}}
    for m in RQ1_MODELS:
        met = RQ1_METRICS[m]
        rows[m] = {l: "{:.3f} / {:.3f} / {:.3f}".format(*macro_component(met, l)) for l in LEVELS}
    return pd.DataFrame(rows).T[LEVELS]

def table_scalar_by_difficulty(section: str, row: str) -> pd.DataFrame:
    rows = {m: {l: RQ1_METRICS[m][section][row][l] for l in LEVELS} for m in RQ1_MODELS}
    return pd.DataFrame(rows).T[LEVELS]

print("Table 2: component matching (accuracy / recall / F1)")
display(table_component_by_difficulty())
print("\nTable 3: exact matching accuracy")
display(table_scalar_by_difficulty("exact matching accuracy", "exact match"))
print("\nTable 4: execution accuracy")
display(table_scalar_by_difficulty("execution accuracy", "execution"))

## 2.3 -- Tables 5-6: execution / exact matching accuracy by database

In [ ]:
def table_by_db(metric: str) -> pd.DataFrame:
    g = rq1_df.groupby(["db_id", "model"])[metric].mean().unstack("model")
    n = rq1_df.groupby("db_id").size().rename("n")
    return g.join(n).sort_values("n", ascending=False)[["n", *RQ1_MODELS]]

print("Table 5: execution accuracy by database")
display(table_by_db("execution"))
print("\nTable 6: exact matching accuracy by database")
display(table_by_db("exact"))

## 2.4 -- Table 7: invalid / unparseable outputs

An output is invalid if it is truncated at the model's `max_new_tokens` (or stopped as a repetition loop)
or finishes without at least 3 words of usable text -- the same `outcome()` classification
`eval/evalpipeline/breakdown.py` uses for the DeepSeek-specific breakdown below, applied here to all three
models.

In [ ]:
def invalid_output_rates() -> pd.DataFrame:
    rows = []
    for m in RQ1_MODELS:
        sub = rq1_df[rq1_df.model == m]
        n_invalid = (sub.outcome != "finished with SQL").sum()
        rows.append({"Model": m, "Invalid outputs": n_invalid, "Invalid rate": f"{n_invalid / len(sub):.1%}"})
    return pd.DataFrame(rows)

invalid_output_rates()

## 2.5 -- Tables 8-9: average generated tokens and inference time

In [ ]:
def tokens_per_example() -> pd.DataFrame:
    g = rq1_df.groupby(["model", "difficulty"])["new_tokens"].mean().unstack("difficulty")
    g["all"] = rq1_df.groupby("model")["new_tokens"].mean()
    return g[["easy", "medium", "hard", "extra", "all"]].round(1)

tokens_per_example()

In [ ]:
def inference_time_summary() -> pd.DataFrame:
    rows = []
    for m, d in RQ1_RUN_DIRS.items():
        gens = pd.DataFrame(load_rows(d))
        total_wall = gens["batch_latency_s"].sum() / GEN_BATCH_SIZE[m]
        rows.append({"Model": m, "Total wall time (s)": round(total_wall, 1),
                     "Avg. time per example (s)": round(total_wall / len(gens), 3)})
    return pd.DataFrame(rows)

inference_time_summary()

## 2.6 -- Tables 10-13: why the reasoning model (DeepSeek) scores low

Reuses `eval.evalpipeline.breakdown.score()` (execution/exact/component-F1 on an arbitrary subset of
generation rows, via a temp-dir call into Spider's own `evaluation.py`) and `.tail_stats()` (compression
ratio of the trailing text, to detect repetition loops) exactly as `RQ1.tex` (tab:deepseek-repetition) does.

In [ ]:
REASONING_MODEL = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
reasoning_rows, reasoning_limit = breakdown.load(RQ1_RUN_DIRS[REASONING_MODEL])
for r in reasoning_rows:
    r["outcome"] = breakdown.outcome(r, reasoning_limit)

OUTCOMES = breakdown.OUTCOMES


def deepseek_outcome_breakdown() -> pd.DataFrame:
    n = len(reasoning_rows)
    out = []
    for o in OUTCOMES + ["all"]:
        part = reasoning_rows if o == "all" else [r for r in reasoning_rows if r["outcome"] == o]
        s = breakdown.score(part)
        tok = sum(r["new_tokens"] for r in part) / len(part) if part else float("nan")
        out.append({"Outcome": o, "Problems (n)": len(part), "Share": f"{len(part) / n:.1%}",
                     "Avg. tokens": round(tok), "Exact match": s["exact"] if s else float("nan"),
                     "Component F1": s["comp_f1"] if s else float("nan")})
    return pd.DataFrame(out)

deepseek_outcome_breakdown()

In [ ]:
def finished_subset_comparison() -> pd.DataFrame:
    finished_ids = {r["id"] for r in reasoning_rows if r["outcome"] == "finished with SQL"}
    rows = []
    for m, d in RQ1_RUN_DIRS.items():
        mrows, _ = breakdown.load(d)
        s = breakdown.score([r for r in mrows if r["id"] in finished_ids])
        rows.append({"Model": m, "Exact match": s["exact"], "Component F1": s["comp_f1"]})
    return pd.DataFrame(rows)

print(f"Table 11: comparison on the {len(set(r['id'] for r in reasoning_rows if r['outcome']=='finished with SQL'))} "
      "questions DeepSeek finished with SQL")
finished_subset_comparison()

In [ ]:
def repetition_loop_stats() -> pd.DataFrame:
    cut = [r for r in reasoning_rows if r["outcome"].startswith("cut")]
    fin = [r for r in reasoning_rows if not r["outcome"].startswith("cut")]
    rows = []
    for name, part in (("Cut off", cut), ("Finished", fin)):
        med, lo_share, recur_share = breakdown.tail_stats(part)
        rows.append({"Generations": name, "n": len(part), "Median compression ratio": round(med, 3),
                     "Ratio < 0.15": f"{lo_share:.1%}", "Last 150 chars repeat >= 3x": f"{recur_share:.1%}"})
    return pd.DataFrame(rows)

repetition_loop_stats()

In [ ]:
def deepseek_unfinished_by_difficulty() -> pd.DataFrame:
    tot = rq1_df[rq1_df.model == REASONING_MODEL].groupby("difficulty").size()
    bad = rq1_df[(rq1_df.model == REASONING_MODEL) & (rq1_df.outcome != "finished with SQL")].groupby("difficulty").size()
    bad = bad.reindex(tot.index, fill_value=0)
    out = pd.DataFrame({"Problems (n)": tot, "Not finished with SQL (n)": bad, "Share": (bad / tot).map(lambda x: f'{x:.1%}')})
    out.loc["all"] = [tot.sum(), bad.sum(), f"{bad.sum() / tot.sum():.1%}"]
    return out.T

deepseek_unfinished_by_difficulty()

## 3. Answer to RQ1 -- a couple of the claims, recomputed

A few of the headline numbers from `RQ1.tex`'s Interpretation and Answer section, recomputed here directly from the fresh run above
(rather than restated), so a reader can check them against the live tables.

In [ ]:
for m in RQ1_MODELS:
    met = RQ1_METRICS[m]
    exact, exe = met["exact matching accuracy"]["exact match"]["all"], met["execution accuracy"]["execution"]["all"]
    print(f"{m}: exact={exact:.3f} execution={exe:.3f} (execution - exact = {exe - exact:+.3f})")

n_invalid = (rq1_df[rq1_df.model == REASONING_MODEL].outcome != "finished with SQL").sum()
n_other_invalid = {m: (rq1_df[rq1_df.model == m].outcome != "finished with SQL").sum() for m in RQ1_MODELS if m != REASONING_MODEL}
print(f"\nDeepSeek invalid-output rate: {n_invalid}/{len(rq1_df[rq1_df.model==REASONING_MODEL])}"
      f" vs. {n_other_invalid}")

# Task 2 -- RQ2: Methodology

RQ2: How do characteristics of reasoning traces (of the reasoning LLM) relate to Text-to-SQL performance?

Full methodology: `RQ2.tex` (Token Count, Information Retention; Spearman's rank correlation with
Benjamini-Hochberg correction as the primary test against both difficulty and 32 per-question performance
submetrics; verified by two regression models -- a logistic regression of exact-match correctness and a
proportional-odds (ordinal) regression of difficulty, each jointly controlling for the other feature --
plus variance inflation factors and a proportional-odds check as diagnostics on those two regressions).
This runs `eval.evalpipeline.reasoning_metrics` (descriptive stats, Spearman correlations, boxplots) and
`eval.evalpipeline.reasoning_verification` (the two regressions and their diagnostics) -- the exact
scripts `RQ2.tex`'s tables are built from -- against the **fresh** DeepSeek run from §1 above (not the
original schema-only Task 1 run `RQ2.tex`'s committed tables were built from), writing to a separate
output directory so nothing here touches the committed `eval/results/task2_analysis/`.

In [ ]:
RQ2_OUT = RESULTS_DIR / "task2_analysis_notebook"
sh([
    sys.executable, "-m", "eval.evalpipeline.reasoning_metrics",
    "--run", str(RQ1_RUN_DIRS[REASONING_MODEL]),
    "--repo-id", REASONING_MODEL,
    "--out", str(RQ2_OUT),
])

In [ ]:
rq2_df = pd.read_csv(RQ2_OUT / "per_question_metrics.csv")
rq2_corr = pd.read_csv(RQ2_OUT / "spearman_correlations.csv")
rq2_desc = pd.read_csv(RQ2_OUT / "descriptive_stats.csv")
rq2_df.head()

# Task 2 -- RQ2: Results

## Table 14: descriptive statistics for Token Count and Information Retention

In [ ]:
rq2_desc

## Table 15: strongest Spearman correlations (Benjamini-Hochberg corrected), top 8 of all tests run

In [ ]:
rq2_corr.sort_values("q_bh").head(8)

In [ ]:
n_sig = (rq2_corr["q_bh"] < 0.05).sum()
print(f"{len(rq2_corr)} correlation tests run; {n_sig} significant at q < 0.05")
rho_diff, p_diff = __import__("scipy").stats.spearmanr(rq2_df["token_count"], rq2_df["difficulty"].map({"easy": 1, "medium": 2, "hard": 3, "extra": 4}), nan_policy="omit")
print(f"Token Count vs. difficulty: rho={rho_diff:.3f}")

## Figure 2: Token Count and Information Retention, by difficulty and by exact-match correctness

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
specs = [("token_count", "Token Count", "difficulty", ["easy", "medium", "hard", "extra"]),
         ("token_count", "Token Count", "exact", [0.0, 1.0]),
         ("retention", "Information Retention", "difficulty", ["easy", "medium", "hard", "extra"]),
         ("retention", "Information Retention", "exact", [0.0, 1.0])]
for ax, (metric, label, by, groups) in zip(axes.flat, specs):
    data = [rq2_df.loc[rq2_df[by] == g, metric].dropna() for g in groups]
    labels = [str(g) if by == "difficulty" else ("incorrect" if g == 0.0 else "correct") for g in groups]
    ax.boxplot(data, tick_labels=labels, showmeans=True)
    ax.set_title(f"{label} by {'difficulty' if by == 'difficulty' else 'exact-match correctness'}")
    ax.set_ylabel(label)
fig.tight_layout()
plt.show()

## Verification: regression controlling for confounds

Runs `eval.evalpipeline.reasoning_verification` against the same `per_question_metrics.csv` §2 above wrote,
into the same `RQ2_OUT` directory. Token Count, Information Retention and difficulty are all correlated
with one another, so a bivariate relationship could just be riding on the other two variables. A logistic
regression predicts exact-match correctness from standardized log Token Count, standardized Information
Retention, and standardized difficulty jointly; a proportional-odds (ordinal) regression predicts
difficulty from the two features jointly.

In [ ]:
sh([
    sys.executable, "-m", "eval.evalpipeline.reasoning_verification",
    "--per-question", str(RQ2_OUT / "per_question_metrics.csv"),
    "--out", str(RQ2_OUT),
])

In [ ]:
logistic_regression = pd.read_csv(RQ2_OUT / "logistic_regression.csv")
ordinal_regression = pd.read_csv(RQ2_OUT / "ordinal_regression.csv")
logistic_regression

In [ ]:
ordinal_regression

## Regression diagnostics

Checks the two assumptions the robustness-check regressions above rely on but don't verify by default:
that the logistic regression's three predictors aren't too collinear (variance inflation factors), and
that the ordinal regression's proportional-odds assumption (a single, constant effect of each predictor
across all three difficulty cutoffs) roughly holds, checked by comparing its pooled coefficients against
three separately-fit cumulative-logit models, one per cutoff.

In [ ]:
vif = pd.read_csv(RQ2_OUT / "vif.csv")
proportional_odds_check = pd.read_csv(RQ2_OUT / "proportional_odds_check.csv")
vif

In [ ]:
proportional_odds_check

# Task 3 -- RQ3: Methodology

RQ3: To what extent does PlanPlay-SQL improve Qwen3-0.6B?

Full methodology: `RQ3.tex` (warm start on gold SQL+plan pairs, verification-based iterative
fine-tuning (VBI-FT), self-play fine-tuning with the SPIN logistic loss trained against a frozen
"opponent" checkpoint; LoRA rank 32 / alpha 64 throughout; `RQ3.tex`'s training-settings table,
reconstructed below as Table 16, lists every stage's hyperparameters, including which checkpoint plays
policy vs. opponent for the self-play rows). `RQ3.tex` scopes this to Qwen3-0.6B only -- there is no
Qwen3-4B extension in the current report, so this notebook does not run one either.

This section drives `train.finetune` (`warm` and `stages` subcommands) and `train.evaluate` directly --
the same modules `train/runs/chain1.sh` / `chain2.sh` / `pp1.out` were produced from -- with every run
renamed with an `_nb` suffix so this notebook never touches the adapters `RQ3.tex`'s tables were
computed from.

**Runtime.** On the 8 GB GPU this repo was developed on, the historical logs (`train/runs/*.log`, `*.out`)
show: each warm start ~25-30 minutes; the gold/VBI/self-play chain (`chain2.sh`) ~35-70 minutes per stage;
the two-round VBI+self-play chain (`pp1.out`) well over an hour. Expect the cells below to take **several
hours in total**. Each is idempotent (guarded by `ensure()`), so interrupting the notebook and restarting
only re-runs whatever did not finish.

In [ ]:
from train.common import RUNS_DIR
TRAIN_ENV = {"SPIDER_DB_DIR": "mock_spider/database"}
print(RUNS_DIR)

## Stage 1 -- Warm start (`chain1.sh` equivalent)

Supervised warm start on 1500 gold `(question, SQL)` pairs, in both prompt formats: `sql` (SQL only) and
`plan` (short plan, then SQL -- the format PlanPlay-SQL actually uses downstream). ~25-30 min each.

In [ ]:
ensure(RUNS_DIR / "e1_sql_nb" / "adapter",
       [sys.executable, "-m", "train.finetune", "warm", "--name", "e1_sql_nb", "--fmt", "sql",
        "--epochs", "3", "--n_train", "1500", "--lr", "2e-4", "--bs", "32", "--val_n", "300"],
       TRAIN_ENV, "(sft_sql ablation)")

ensure(RUNS_DIR / "e1_plan_nb" / "adapter",
       [sys.executable, "-m", "train.finetune", "warm", "--name", "e1_plan_nb", "--fmt", "plan",
        "--epochs", "3", "--n_train", "1500", "--lr", "2e-4", "--bs", "32", "--val_n", "300"],
       TRAIN_ENV, "(the warm-start checkpoint every later stage builds on)")

## Stage 2 -- Supervised control / VBI-FT / self-play (higher LR) (`chain2.sh` equivalent)

Three independent stages, each starting from `e1_plan_nb/adapter`, each trained/evaluated on the same 1500
training questions held out of the warm start (`--exclude_n 1500 --n_round 1500 --rounds 1`):

- `gold`: a further 1500 **gold** SQL+plan pairs (the supervised control Table 16/18 compare the other
  stages against).
- `vbi`: verification-based iterative fine-tuning -- sample 4 completions/question, keep only
  execution-verified positives, restrict to questions where not every sample was already correct
  (`--hard_only`).
- `sp` (higher LR): self-play fine-tuning (SPIN logistic loss) against the warm-start checkpoint as the
  frozen opponent, at the higher of the two learning rates `RQ3.tex`'s training-settings table reports
  (row F: the opponent is the same warm-start checkpoint, since round 1 has no earlier checkpoint to play
  that role).

Each omits `--init_acc`, so it spends an extra ~7 minutes evaluating `e1_plan_nb` on the full 707-question
validation set itself (the original chain passed a precomputed `--init_acc` to skip this; skipping
that optimization keeps this notebook simpler and self-contained).

In [ ]:
COMMON_S2 = ["--fmt", "plan", "--init", str(RUNS_DIR / "e1_plan_nb" / "adapter"),
             "--exclude_n", "1500", "--n_round", "1500", "--rounds", "1", "--bs", "32"]

ensure(RUNS_DIR / "c1_gold_nb" / "r1_gold",
       [sys.executable, "-m", "train.finetune", "stages", "--name", "c1_gold_nb", "--stages", "gold",
        "--lr_vbi", "1e-4", *COMMON_S2], TRAIN_ENV, "(supervised control)")

ensure(RUNS_DIR / "c1_vbi_nb" / "r1_vbi",
       [sys.executable, "-m", "train.finetune", "stages", "--name", "c1_vbi_nb", "--stages", "vbi",
        "--k", "4", "--hard_only", "--lr_vbi", "1e-4", *COMMON_S2], TRAIN_ENV, "(VBI-FT)")

ensure(RUNS_DIR / "c1_sp_nb" / "r1_sp",
       [sys.executable, "-m", "train.finetune", "stages", "--name", "c1_sp_nb", "--stages", "sp",
        "--k_sp", "4", "--lr_sp", "1e-4", "--lam", "0.1", "--epochs", "2", *COMMON_S2],
       TRAIN_ENV, "(self-play, higher LR)")

## Stage 3 -- VBI-FT then self-play, lower LR (`pp1.out` equivalent)

Two rounds of VBI-FT followed by self-play (SPIN) at the lower of the two learning rates, sampling 800
fresh training questions per round (no `--exclude_n`, so the pool includes the warm-start's own 1500
questions, exactly as the original `pp1` run did). This is the longest stage.

In [ ]:
ensure(RUNS_DIR / "pp1_nb" / "r1_sp",
       [sys.executable, "-m", "train.finetune", "stages", "--name", "pp1_nb", "--fmt", "plan",
        "--init", str(RUNS_DIR / "e1_plan_nb" / "adapter"), "--stages", "vbi,sp",
        "--rounds", "2", "--n_round", "800", "--bs", "32"],
       TRAIN_ENV, "(VBI-FT -> self-play, lower LR, round 1)")

# Task 3 -- RQ3: Results

## Table 16 (reconstructed): training settings actually used, read back from each stage's own `result.json` / `history.json`

Rather than retype the hyperparameters, this reads back the `args` dict every stage already wrote to disk,
so this table can never silently drift from what was actually run. Matches `RQ3.tex`'s lettered A-F rows,
including the Policy init / Opponent split for the self-play rows: `pp1_nb`'s first round runs VBI-FT
(`r1_vbi`, row D) then self-play (`r1_sp`, row E) back to back on the same fresh 800-question sample, so
both are read back separately below rather than only the final self-play number.

In [ ]:
def stage_record(run_name: str, hist_key: str) -> dict:
    hist = json.loads((RUNS_DIR / run_name / "history.json").read_text())
    args, row = hist["args"], next(h for h in hist["history"] if h["stage"] == hist_key)
    return args, row

rows = []
warm_args = json.loads((RUNS_DIR / "e1_plan_nb" / "result.json").read_text())["args"]
rows.append({"Stage": "A: Warm start (plan format, 1500 gold)", "Policy init": "Qwen3-0.6B", "Opponent": "-",
             "Training items": "1500 gold SQL+plan pairs", "Samples/question (k)": "-",
             "Learning rate": warm_args["lr"], "Epochs": warm_args["epochs"], "Batch size": warm_args["bs"],
             "Rounds": 1})

args, row = stage_record("c1_gold_nb", "r1_gold")
rows.append({"Stage": "B: A + 1500 further gold (supervised control)", "Policy init": "Warm start", "Opponent": "-",
             "Training items": f"{row['n_items']} gold SQL+plan pairs", "Samples/question (k)": "-",
             "Learning rate": args["lr_vbi"], "Epochs": args["epochs"], "Batch size": args["bs"], "Rounds": 1})

args, row = stage_record("c1_vbi_nb", "r1_vbi")
rows.append({"Stage": "C: B + VBI-FT (1500 q, hard only)", "Policy init": "Warm start", "Opponent": "-",
             "Training items": f"{row['n_items']} verified positives (of {args['n_round'] * args['k']} sampled)",
             "Samples/question (k)": args["k"], "Learning rate": args["lr_vbi"], "Epochs": args["epochs"],
             "Batch size": args["bs"], "Rounds": 1})

# Row D: pp1_nb's own round-1 VBI-FT pass (a plain VBI-FT step, not self-play itself -- it just produces
# the slightly-improved policy checkpoint that row E's self-play step then trains against the opponent).
args_d, row_d = stage_record("pp1_nb", "r1_vbi")
rows.append({"Stage": "D: self-play (lower LR), step 1/2: internal VBI-FT (own 800 q)", "Policy init": "Warm start",
             "Opponent": "-",
             "Training items": f"{row_d['n_items']} verified positives (of {args_d['n_round'] * args_d['k']} sampled)",
             "Samples/question (k)": args_d["k"], "Learning rate": args_d["lr_vbi"], "Epochs": args_d["epochs"],
             "Batch size": args_d["bs"], "Rounds": f"1 of {args_d['rounds']} planned"})

# Row E: the actual self-play preference step. Policy init is row D's own checkpoint; the opponent is
# necessarily the warm start, since M = {warm start, D's checkpoint} is all run_stages() has seen by here.
args_e, row_e = stage_record("pp1_nb", "r1_sp")
rows.append({"Stage": "E: self-play (lower LR), step 2/2: preference training",
             "Policy init": f"D's checkpoint (val. acc. {row_d['acc']:.3f})", "Opponent": "Warm start",
             "Training items": f"{row_e['n_pairs']} verified preference pairs (of {args_e['n_round'] * args_e['k_sp']} sampled)",
             "Samples/question (k)": args_e["k_sp"], "Learning rate": args_e["lr_sp"], "Epochs": args_e["epochs"],
             "Batch size": args_e["bs"], "Rounds": f"1 of {args_e['rounds']} planned"})

# Row F: self-play, higher LR, round 1 of a single-round run -- policy and opponent are both the warm
# start, since round 1 has no earlier checkpoint to play the opponent role (the degenerate case).
args_f, row_f = stage_record("c1_sp_nb", "r1_sp")
rows.append({"Stage": "F: self-play (higher LR), round 1", "Policy init": "Warm start",
             "Opponent": "Warm start (same checkpoint; round 1 has no earlier opponent)",
             "Training items": f"{row_f['n_pairs']} verified preference pairs (of {args_f['n_round'] * args_f['k_sp']} sampled)",
             "Samples/question (k)": args_f["k_sp"], "Learning rate": args_f["lr_sp"], "Epochs": args_f["epochs"],
             "Batch size": args_f["bs"], "Rounds": 1})

pd.DataFrame(rows)

## Table 18: validation accuracy after each stage

The two zero-shot baseline rows (marked `*` in `RQ3.tex`, a 300-question subset of the same validation
set) are computed inline here. Every other row is the `acc` field each stage already wrote to
`history.json` / `result.json` when it chose its best checkpoint on the full 707-question validation set.

In [ ]:
from train.common import build_splits, eval_examples, load_base, MAX_NEW

def zero_shot_baselines() -> dict:
    py_code = '''
import json, random
from train.common import build_splits, eval_examples, load_base, MAX_NEW
tok, base = load_base(checkpoint=False)
_, val, _ = build_splits(tokenizer=tok)
val = random.Random(0).sample(val, min(300, len(val)))
out = {}
for fmt in ("sql", "plan"):
    r = eval_examples(base, tok, val, fmt, MAX_NEW[fmt])
    out[fmt] = r["acc"]
print(json.dumps(out))
'''
    res = subprocess.run([sys.executable, "-c", py_code], check=True, capture_output=True, text=True,
                          env={**os.environ, **TRAIN_ENV}, cwd=REPO_ROOT)
    return json.loads(res.stdout.strip().splitlines()[-1])

zs_marker = RUNS_DIR / "zero_shot_baselines_nb.json"
if zs_marker.exists():
    zs = json.loads(zs_marker.read_text())
else:
    zs = zero_shot_baselines()
    zs_marker.write_text(json.dumps(zs))
zs

In [ ]:
def validation_accuracy_by_stage() -> pd.DataFrame:
    rows = [
        {"Stage": "Base model, original prompt*", "Validation accuracy": zs["sql"]},
        {"Stage": "Base model, plan prompt, no training*", "Validation accuracy": zs["plan"]},
        {"Stage": "Warm start on 1500 gold questions (plan format)",
         "Validation accuracy": json.loads((RUNS_DIR / 'e1_plan_nb' / 'result.json').read_text())["history"][-1]["acc"]},
        {"Stage": "+ 1500 further gold questions (supervised control)",
         "Validation accuracy": stage_record('c1_gold_nb', 'r1_gold')[1]["acc"]},
        {"Stage": "+ VBI-FT on the same 1500 questions (hard questions only)",
         "Validation accuracy": stage_record('c1_vbi_nb', 'r1_vbi')[1]["acc"]},
        {"Stage": "+ self-play, lower LR", "Validation accuracy": stage_record('pp1_nb', 'r1_sp')[1]["acc"]},
        {"Stage": "+ self-play, higher LR", "Validation accuracy": stage_record('c1_sp_nb', 'r1_sp')[1]["acc"]},
    ]
    return pd.DataFrame(rows)

stage_accuracy_table = validation_accuracy_by_stage()
stage_accuracy_table

## Table 17: original vs. adapted Qwen3-0.6B on the full 1034-question dev set (mock-DB execution)

Per the methodology above, the checkpoint selected on validation data (not necessarily the end of the full
chain) is the one reported as "the adapted model" -- `RQ3.tex` itself does this: its dev-results table's
"Adapted" row is the supervised-control checkpoint (`gold_control`), which had the best validation accuracy
of all the ablations in its own validation-accuracy table. We pick the best `*_nb` checkpoint the same way, automatically.

Both the baseline and the adapted model are evaluated **in a fresh subprocess** with `SPIDER_DB_DIR` set, so
the mock databases (not the schema-only ones used for RQ1/RQ2 earlier in this kernel) are what gets scored.

In [ ]:
trained_stage_names = {
    "Warm start": ("e1_plan_nb", None),
    "+ supervised control": ("c1_gold_nb", "r1_gold"),
    "+ VBI-FT": ("c1_vbi_nb", "r1_vbi"),
    "+ self-play (lower LR)": ("pp1_nb", "r1_sp"),
    "+ self-play (higher LR)": ("c1_sp_nb", "r1_sp"),
}
accs = stage_accuracy_table.set_index("Stage")["Validation accuracy"]
best_label = max(
    ["Warm start on 1500 gold questions (plan format)", "+ 1500 further gold questions (supervised control)",
     "+ VBI-FT on the same 1500 questions (hard questions only)", "+ self-play, lower LR", "+ self-play, higher LR"],
    key=lambda l: accs[l],
)
best_key = {
    "Warm start on 1500 gold questions (plan format)": "Warm start",
    "+ 1500 further gold questions (supervised control)": "+ supervised control",
    "+ VBI-FT on the same 1500 questions (hard questions only)": "+ VBI-FT",
    "+ self-play, lower LR": "+ self-play (lower LR)",
    "+ self-play, higher LR": "+ self-play (higher LR)",
}[best_label]
run_name, sub = trained_stage_names[best_key]
best_adapter = RUNS_DIR / run_name / "adapter" if sub is None else RUNS_DIR / run_name / sub
print(f"Best checkpoint by validation accuracy ({accs[best_label]:.4f}): {best_adapter}")

In [ ]:
BASE_RQ1_DIR = RQ1_RUN_DIRS["Qwen/Qwen3-0.6B"]
BASE_NB_DIR = RESULTS_DIR / "Qwen__Qwen3-0.6B" / "base_task1_nb"
ADAPTED_NB_NAME = "adapted_nb"

def rescore_baseline_on_mock_db():
    code_str = f"""
import shutil
from pathlib import Path
from eval.evalpipeline.evaluate import evaluate_run
src = Path(r"{BASE_RQ1_DIR}")
dst = Path(r"{BASE_NB_DIR}")
dst.mkdir(parents=True, exist_ok=True)
for f in ("generations.jsonl", "gold.txt", "pred.txt", "config_used.yaml"):
    shutil.copy(src / f, dst / f)
evaluate_run(dst, "all")
print("done")
"""
    py(code_str, TRAIN_ENV)

def run_adapted_on_dev():
    code_str = f"""
from train.evaluate import run_one
run_one({ADAPTED_NB_NAME!r}, r"{best_adapter}", "plan")
"""
    py(code_str, TRAIN_ENV)

if not (BASE_NB_DIR / "metrics.json").exists():
    rescore_baseline_on_mock_db()
else:
    print(f"[skip] {BASE_NB_DIR / 'metrics.json'} already exists.")
adapted_dir = RESULTS_DIR / "Qwen__Qwen3-0.6B" / ADAPTED_NB_NAME
if not (adapted_dir / "metrics.json").exists():
    run_adapted_on_dev()

In [ ]:
def original_vs_adapted_dev_results() -> pd.DataFrame:
    base_met = json.loads((BASE_NB_DIR / "metrics.json").read_text())
    adapt_met = json.loads((adapted_dir / "metrics.json").read_text())
    rows = []
    for label, met in (("Original (fresh RQ1 run, mock DB)", base_met), ("Adapted (PlanPlay-SQL, mock DB)", adapt_met)):
        for lvl in LEVELS:
            a, r, f = macro_component(met, lvl)
            rows.append({
                "Model": label, "Difficulty": lvl,
                "Component matching (acc/rec/F1)": f"{a:.3f} / {r:.3f} / {f:.3f}",
                "Exact matching": met["exact matching accuracy"]["exact match"][lvl],
                "Execution (mock DB)": met["execution accuracy"]["execution"][lvl],
            })
    return pd.DataFrame(rows)

original_vs_adapted_dev_results()

## Qualitative examples

Scans the fresh dev-set generations for one question the adapted model fixed (original wrong, adapted
right by mock-DB execution match) and one it broke (the reverse), the same comparison `RQ3.tex`
illustrates with two fixed examples -- but found fresh from this run rather than restated.

In [ ]:
from eval.evalpipeline.reasoning_metrics import load_rows as _load_rows

base_rows = {r["id"]: r for r in _load_rows(BASE_NB_DIR)}
adapt_rows = {r["id"]: r for r in _load_rows(adapted_dir)}

# per-question execution correctness on the mock DB, recovered with a fresh subprocess (see the DB_DIR note above)
code_str = f'''
import json
from eval.evalpipeline.reasoning_metrics import load_rows, per_question_eval
base = {{s["id"]: s for s in per_question_eval(load_rows(r"{BASE_NB_DIR}"))}}
adapt = {{s["id"]: s for s in per_question_eval(load_rows(r"{adapted_dir}"))}}
fixed = [i for i in base if base[i]["execution"] == 0 and adapt.get(i, {{}}).get("execution") == 1]
broke = [i for i in base if base[i]["execution"] == 1 and adapt.get(i, {{}}).get("execution") == 0]
print(json.dumps({{"fixed": fixed[:1], "broke": broke[:1]}}))
'''
res = subprocess.run([sys.executable, "-c", code_str], check=True, capture_output=True, text=True,
                      env={**os.environ, **TRAIN_ENV}, cwd=REPO_ROOT)
picks = json.loads(res.stdout.strip().splitlines()[-1])

for label, ids in (("Fixed by the adapted model", picks["fixed"]), ("Broken by the adapted model", picks["broke"])):
    print(f"=== {label} ===")
    for qid in ids:
        b, a = base_rows[qid], adapt_rows[qid]
        print(f"Q: {b['question']}  (db: {b['db_id']})")
        print(f"  gold:     {' '.join(b['gold'].split())}")
        print(f"  original: {b['pred']}")
        print(f"  adapted:  {a['pred']}")
    print()

## Limitations

See `RQ1.tex` / `RQ2.tex` / `RQ3.tex`'s own Limitations sections -- mock-database execution accuracy,
single run/seed, schema-only vs. mock-DB comparability, and the component-matching macro-average all
apply identically to the numbers recomputed in this notebook. In addition: this run used a different GPU
and a different point in the RNG stream than the run the report was written from, so exact figures are
expected to differ in the last one or two decimal places while the qualitative conclusions (which
checkpoint is best, which stage drives the gain, that self-play at a high learning rate collapses) should
match.